# PanoraDent AI — entraînement sur Colab (machine de repli)

**Le code vit dans le dépôt, seul le calcul tourne ici.** Ce carnet appelle les
mêmes scripts que la machine du cabinet (`training/README.md`) : seule la carte
change. À utiliser si la carte locale ne suffit pas (Colab T4 : 16 Go de mémoire,
précision mixte fonctionnelle).

Données : DENTEX (Hugging Face, révision figée), **CC BY-NC-SA 4.0** — usage
académique et non commercial ; les poids entraînés en héritent.

## 1. Vérifier la carte

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch; print(torch.__version__, torch.cuda.is_available())

## 2. Dépôt et dépendances

Le dépôt est privé : utiliser un jeton GitHub personnel en lecture seule, saisi au moment du clonage, jamais écrit dans le carnet.

In [ ]:
%cd /content
from getpass import getpass
jeton = getpass("Jeton GitHub (lecture seule) : ")
!git clone --depth 1 --branch feat/sprint6-registre-modeles https://{jeton}@github.com/STAGE-M2-ORION/PanoraDent-AI.git panoradent
del jeton
%cd /content/panoradent
!pip install -q -r training/requirements.txt

## 3. Télécharger et préparer DENTEX

Archives vérifiées (sha256), puis deux jeux — « dents » et « anomalies » — et le test externe, avec un seul découpage fixé par la graine.

In [ ]:
!python training/telecharger_dentex.py --dest data/dentex
!python training/prepare_dataset.py --dentex data/dentex --out data/yolo

## 4. Entraîner les deux modèles

Sur une T4, des lots de 8 tiennent en 1024 px avec la précision mixte. Reprise après une coupure de session : `--reprendre runs/<nom>/weights/last.pt`.

In [ ]:
!python training/train.py --data data/yolo/dents/data.yaml     --name dents-v1     --batch 8 --device 0
!python training/train.py --data data/yolo/anomalies/data.yaml --name anomalies-v1 --batch 8 --device 0

## 5. Évaluer

La validation autant que nécessaire ; le test interne puis le test externe **une seule fois**, à la fin.

In [ ]:
for nom in ("dents-v1", "anomalies-v1"):
    !python training/evaluate.py --weights runs/{nom}/weights/best.pt --split val

In [ ]:
!python training/evaluate.py --weights runs/dents-v1/weights/best.pt --split test
!python training/evaluate.py --weights runs/anomalies-v1/weights/best.pt --split test
!python training/evaluate.py --weights runs/anomalies-v1/weights/best.pt --data data/yolo/externe/data.yaml --split test

## 6. Récupérer les runs

Vers Drive : la session Colab sera perdue. Puis, **sur la machine qui héberge la stack**, publier la paire (`training/publish_model.py`), produire le rapport (`training/rapport_evaluation.py`) et activer la version depuis l'administration.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/panoradent/runs
!cp -r runs/dents-v1 runs/anomalies-v1 data/yolo/dents/split.json /content/drive/MyDrive/panoradent/runs/